# Pályázati felhívás kockázati elemző – futtatható demo notebook

Ez a notebook bemutatja a beadandó pipeline fő lépéseit: PDF szövegkinyerés, ZSC kategorizálás, intent felismerés, LLM összefoglaló, kockázati pontszámítás és adatbázisba mentés.

In [26]:
from pathlib import Path
import sys
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(BASE_DIR))
BASE_DIR

WindowsPath('C:/Users/Kitti/OneDrive/Desktop/Új mappa/palyazat_fixed')

## 1. PDF-ek listázása

In [27]:
pdf_dir = BASE_DIR / "data" / "pdfs"
pdf_files = sorted(pdf_dir.glob("*.pdf"))[:3]
pdf_files

[WindowsPath('C:/Users/Kitti/OneDrive/Desktop/Új mappa/palyazat_fixed/data/pdfs/ginop-plusz-121-felhivas.pdf'),
 WindowsPath('C:/Users/Kitti/OneDrive/Desktop/Új mappa/palyazat_fixed/data/pdfs/ginop-plusz-122-22-felhivas.pdf'),
 WindowsPath('C:/Users/Kitti/OneDrive/Desktop/Új mappa/palyazat_fixed/data/pdfs/ginop-plusz-123-felhivas.pdf')]

## 2. Egy PDF szövegének kinyerése és mezőkinyerés

In [28]:
from app.extractor import read_pdf_text, extract_fields
text = read_pdf_text(pdf_files[0])
fields = extract_fields(text)
fields

{'call_code': 'GINOP Plusz-1.2.1-21',
 'title': 'A mikro-, kis- és középvállalkozások modern üzleti és termelési kihívásokhoz való alkalmazkodását segítő fejlesztések támogatása',
 'advance_percent': 100,
 'advance_max': 629300000,
 'own_fund_percent': 30,
 'own_fund_required': 'igen',
 'consortium_allowed': 'nem',
 'support_logic_text': 'A visszatérítendő támogatás bizonyos feltételek teljesülése mellett részben vagy egészben vissza nem',
 'support_type': 'feltételesen vissza nem térítendő',
 'project_duration_months': 24,
 'max_support': 629300000,
 'min_support': 10000000,
 'total_budget_huf': 121100000000,
 'submission_start': None,
 'submission_end': None,
 'beneficiary_text': '\uf0b7 amelyek rendelkeznek legalább 1 lezárt, teljes üzleti évvel \uf0b7 amelyek éves átlagos statisztikai állományi létszáma a támogatási kérelem benyújtását megelőző lezárt, teljes, de 2019-nél nem korábbi üzleti évben legalább 3 fő volt, \uf0b7 amelyek Magyarország területén székhellyel rendelkező kettő

## 3. ZSC kategorizálás

In [29]:
from app.zsc_classifier import classify_text
category = classify_text(text)
category

'kutatás-fejlesztés'

## 4. Intent recognizer

In [30]:
from app.intent_model import IntentRecognizer
intent_model = IntentRecognizer()
intent_model.predict("Milyen kockázatok vannak ebben a felhívásban?")

np.str_('risk_analysis')

## 5. Kockázati pontszámítás

In [31]:
from app.risk_model import compute_risk
risk = compute_risk(fields)
risk

(13,
 'magas',
 {'előleg': {'pont': 3,
   'megjegyzés': '100% – magas előleg, jelentős visszakövetelési kockázat'},
  'futamidő': {'pont': 2, 'megjegyzés': '24 hónap – közepes futamidő'},
  'max_támogatás': {'pont': 3, 'megjegyzés': '629 M Ft – nagy összeg'},
  'projektek_száma': {'pont': 1,
   'megjegyzés': 'ismeretlen – nem értékelhető (1 pont büntetés)'},
  'támogatás_típusa': {'pont': 3,
   'megjegyzés': 'vissza nem térítendő – nincs visszafizetési biztosíték, EU-szabálytalanság esetén kötelezettségszegési eljárás'},
  'önerő': {'pont': 1, 'megjegyzés': '30% önerő – közepes elköteleződés'},
  'konzorcium': {'pont': 0,
   'megjegyzés': 'konzorcium nem lehetséges – egy kedvezményezett, egyértelmű felelősség'}})

## 6. LLM összefoglaló – Mistral/Ollama
Ez a cella akkor fut jól, ha az Ollama és a Mistral modell elérhető.

In [32]:
from app.llm_summary import generate_stable_summary, is_ollama_available
is_ollama_available()
# summary = generate_stable_summary(text)
# print(summary[:2000])

(True, 'mistral:latest, tinyllama:latest')

## 7. Teljes pipeline futtatása
A parancs a projekt gyökeréből futtatható.

In [33]:
# !PDF_LIMIT=3 python -m app.ingest
# !python -m app.monitoring